# Benchmark OCR sur GPU Kaggle

Seule la **Phase 1** (inference OCR) a besoin d'un GPU. Les evaluations
(TEDS, fusion, champs) tournent en quelques secondes sur ton PC : on les
garde en local, on ne rapatrie que `results/`.

## Avant de lancer

1. **Settings -> Accelerator -> GPU T4 x2** (ou P100)
2. **Settings -> Internet -> On** (sinon ni `pip install` ni `git clone`)
3. **Add Input -> Datasets** : ajoute ton dataset d'images
4. Verifie que le depot GitHub est a jour (`git push` depuis ton PC)

Puis **Save Version -> Save & Run All (Commit)** : le notebook tourne en
arriere-plan, tu peux fermer l'onglet. Sinon la session meurt avec la page.

## 1. Verifier le GPU

In [ ]:
!nvidia-smi
import sys, os
print(sys.version)

## 2. Recuperer le code

Le depot ne contient que le code : les images vivent dans le dataset Kaggle,
et `results/` est ignore par git.

In [ ]:
REPO = "https://github.com/Halfaa/PTITOcr"

%cd /kaggle/working
!rm -rf PTITOcr
!git clone --depth 1 $REPO
%cd /kaggle/working/PTITOcr
!ls

## 3. Localiser les images du dataset

Kaggle monte tes datasets en lecture seule sous `/kaggle/input/<nom>/`.
Cette cellule trouve le dossier toute seule plutot que de coder le chemin
en dur, qui change des que tu renommes le dataset.

In [ ]:
from pathlib import Path

# Le dossier d'images = celui qui contient le plus de page_*.png
candidates = {}
for png in Path("/kaggle/input").rglob("page_*.png"):
    candidates[png.parent] = candidates.get(png.parent, 0) + 1

for folder, n in sorted(candidates.items(), key=lambda kv: -kv[1]):
    print(f"{n:>4} images   {folder}")

assert candidates, "Aucune image trouvee : as-tu ajoute le dataset via Add Input ?"
IMAGES_DIR = max(candidates, key=candidates.get)
print("\n-> retenu :", IMAGES_DIR)

## 4. Installer les dependances

**numpy < 2 est imperatif** : paddlex ne supporte pas NumPy 2.x, et l'erreur
qu'il leve ("a dependency error occurred during pipeline creation") ne nomme
ni le paquet ni l'exception reelle. `src/env_check.py` verifie ce plafond au
demarrage et echoue en une seconde plutot qu'apres le chargement du modele.

Redemarre le noyau apres cette cellule si numpy etait deja importe.

In [ ]:
!pip install -q "numpy<2" "opencv-python<4.12"
!pip install -q paddlepaddle-gpu
!pip install -q "paddleocr[doc-parser]>=3.4.0"

import importlib.metadata as md
for pkg in ("numpy", "paddlepaddle-gpu", "paddleocr", "paddlex"):
    try:
        print(f"{pkg:<18}{md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<18}absent")

In [ ]:
# Le GPU est-il vu par Paddle ? Si non, l'inference tombera sur le CPU
# SANS erreur -- et tu ne t'en apercevras qu'au temps par image.
import paddle
paddle.utils.run_check()
print("GPU compile :", paddle.is_compiled_with_cuda())
print("GPU dispo   :", paddle.device.get_device())

## 5. Lancer le benchmark

Le script ecrit dans `results/<--output>/` et reecrit son rapport apres
chaque image : une interruption ne coute qu'une page. Relancer reutilise
les pages deja faites.

In [ ]:
MODEL = "paddleocr-vl"          # paddleocr-vl | ppstructure | hunyuan
RUN_NAME = "multipage_preproc2_paddleocr"

%cd /kaggle/working/PTITOcr
!python scripts/bench_deformed.py \
    --model $MODEL \
    --images-dir "$IMAGES_DIR" \
    --output $RUN_NAME

## 6. Rapatrier les resultats

Tout ce qui se trouve dans `/kaggle/working` est telechargeable depuis
l'onglet **Output** a la fin du run. On zippe pour n'avoir qu'un fichier.

Puis, **sur ton PC**, decompresse dans `MedicalRecord/results/` et lance :

```powershell
python src\eval_teds.py            --run multipage_preproc2_paddleocr
python src\eval_multipage_merge.py --run multipage_preproc2_paddleocr
python src\eval_fields.py --runs multipage_paddleocr_vl multipage_preproc2_paddleocr
```

In [ ]:
import shutil
from pathlib import Path

src = Path("/kaggle/working/PTITOcr/results") / RUN_NAME
assert src.exists(), f"{src} absent : le benchmark a-t-il tourne ?"

out = shutil.make_archive(f"/kaggle/working/{RUN_NAME}", "zip", src.parent, RUN_NAME)
print(out, f"{Path(out).stat().st_size / 1e6:.1f} Mo")

# Le depot clone pese pour rien dans l'Output : on le retire.
shutil.rmtree("/kaggle/working/PTITOcr", ignore_errors=True)
print("\nContenu de /kaggle/working :")
for p in sorted(Path("/kaggle/working").iterdir()):
    print(" ", p.name)

---
## Variante HunyuanOCR

Environnement incompatible avec Paddle (transformers >= 5.13) : fais-en un
notebook separe plutot que de melanger les deux dans la meme session.

**Attention licence.** La licence communautaire Tencent exclut l'Union
europeenne, le Royaume-Uni et la Coree du Sud de son Territoire, et Kaggle
ne permet pas de choisir la region de la machine. PaddleOCR-VL, en Apache
2.0, n'a aucune de ces contraintes. A trancher avec ton encadrante avant de
lancer Hunyuan ici.

In [ ]:
# !pip install -q "transformers>=5.13.0" accelerate pillow
# !python scripts/bench_deformed.py --model hunyuan \
#     --images-dir "$IMAGES_DIR" --output multipage_preproc2_hunyuan